<a href="https://colab.research.google.com/github/pachir1su/file_swordmaster/blob/main/ALL_swordmaster.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install gradio pypdf pydub reportlab mutagen

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.4/194.4 kB 15.4 MB/s eta 0:00:00


In [4]:
import gradio as gr
from pypdf import PdfReader, PdfWriter
from pydub import AudioSegment
from pydub.silence import split_on_silence
import os
import io
import zipfile
import mutagen

from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import A4
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.cidfonts import UnicodeCIDFont

pdfmetrics.registerFont(UnicodeCIDFont('HYGothic-Medium'))

# ==========================================
# 공통 유틸
# ==========================================
def parse_page_string(page_string, total_pages):
    pages = set()
    if not page_string:
        return []
    for part in page_string.split(','):
        part = part.strip()
        if '-' in part:
            start, end = map(int, part.split('-'))
            pages.update(range(start, end + 1))
        else:
            pages.add(int(part))
    return sorted([p for p in pages if 1 <= p <= total_pages])

def parse_time_to_ms(time_str):
    if not time_str or str(time_str).strip() in ["", "0"]:
        return 0
    time_str = str(time_str).strip()
    try:
        if ':' in time_str:
            parts = time_str.split(':')
            if len(parts) == 2:
                return int((int(parts[0]) * 60 + float(parts[1])) * 1000)
            elif len(parts) == 3:
                return int((int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])) * 1000)
        else:
            return int(float(time_str) * 1000)
    except Exception:
        raise ValueError(f"잘못된 시간 형식: {time_str}")
    return 0

# ==========================================
# 1. PDF 탭 — 분할 / 병합 / 보안·워터마크
# ==========================================
def process_pdf_split(pdf_file, page_string, custom_name):
    if pdf_file is None:
        return None, "PDF 파일을 업로드해주세요.", gr.update(interactive=False)
    try:
        reader = PdfReader(pdf_file.name)
        total_pages = len(reader.pages)
        target_pages = parse_page_string(page_string, total_pages)
        if not target_pages:
            return None, f"유효하지 않은 입력입니다. (총 {total_pages}페이지)", gr.update(interactive=False)

        writer = PdfWriter()
        for p in target_pages:
            writer.add_page(reader.pages[p - 1])

        name_only, ext = os.path.splitext(os.path.basename(pdf_file.name))
        if custom_name and custom_name.strip():
            output_path = f"{custom_name.strip()}{ext}"
        else:
            safe_page_str = page_string.replace(' ', '').replace(',', '_')
            output_path = f"{name_only}_pages_{safe_page_str}{ext}"

        with open(output_path, "wb") as f:
            writer.write(f)
        return output_path, f"성공! 추출 페이지: {target_pages}", gr.update(value=output_path, interactive=True)
    except Exception as e:
        return None, f"오류: {str(e)}", gr.update(interactive=False)


def process_pdf_merge(pdf_files, password, custom_name):
    if not pdf_files:
        return None, "합칠 PDF 파일들을 업로드해주세요.", gr.update(interactive=False)
    try:
        writer = PdfWriter()
        for pdf in pdf_files:
            reader = PdfReader(pdf.name)
            if reader.is_encrypted:
                if password:
                    reader.decrypt(password)
                else:
                    return None, "잠긴 파일이 있습니다. 비밀번호를 입력해주세요.", gr.update(interactive=False)
            for page in reader.pages:
                writer.add_page(page)

        name_only, _ = os.path.splitext(os.path.basename(pdf_files[0].name))
        if custom_name and custom_name.strip():
            output_path = f"{custom_name.strip()}.pdf"
        else:
            count_tag = f"_and_{len(pdf_files)-1}others" if len(pdf_files) > 1 else ""
            output_path = f"{name_only}{count_tag}_merged.pdf"

        with open(output_path, "wb") as f:
            writer.write(f)
        return output_path, f"{len(pdf_files)}개 파일 병합 완료!", gr.update(value=output_path, interactive=True)
    except Exception as e:
        return None, f"오류: {str(e)}", gr.update(interactive=False)


def process_pdf_security(pdf_file, new_password, watermark_text, custom_name):
    if pdf_file is None:
        return None, "PDF 파일을 업로드해주세요.", gr.update(interactive=False)
    try:
        reader = PdfReader(pdf_file.name)
        writer = PdfWriter()

        watermark_reader = None
        if watermark_text and watermark_text.strip():
            packet = io.BytesIO()
            can = canvas.Canvas(packet, pagesize=A4)
            can.setFont("HYGothic-Medium", 60)
            can.setFillColorRGB(0.5, 0.5, 0.5, alpha=0.3)
            can.translate(300, 400)
            can.rotate(45)
            can.drawCentredString(0, 0, watermark_text)
            can.save()
            packet.seek(0)
            watermark_reader = PdfReader(packet)
            watermark_page = watermark_reader.pages[0]

        for page in reader.pages:
            if watermark_reader:
                page.merge_page(watermark_page)
            writer.add_page(page)

        if new_password and new_password.strip():
            writer.encrypt(new_password)

        name_only, _ = os.path.splitext(os.path.basename(pdf_file.name))
        if custom_name and custom_name.strip():
            output_path = f"{custom_name.strip()}.pdf"
        else:
            output_path = f"{name_only}_secured.pdf"

        with open(output_path, "wb") as f:
            writer.write(f)
        return output_path, "보안 처리 완료!", gr.update(value=output_path, interactive=True)
    except Exception as e:
        return None, f"오류: {str(e)}", gr.update(interactive=False)


def handle_pdf(mode, pdf_single, page_string,
               pdf_multi, merge_password,
               security_pdf, security_pw, watermark_text,
               txt_files, txt_output_mode,
               custom_name):
    if mode == "분할":
        return process_pdf_split(pdf_single, page_string, custom_name)
    elif mode == "병합 / 잠금해제":
        return process_pdf_merge(pdf_multi, merge_password, custom_name)
    elif mode == "보안 / 워터마크":
        return process_pdf_security(security_pdf, security_pw, watermark_text, custom_name)
    elif mode == "텍스트 → PDF":
        return process_txt_to_pdf(txt_files, txt_output_mode, custom_name)
    return None, "기능을 선택해주세요.", gr.update(interactive=False)


# ==========================================
# 텍스트 → PDF 변환
# ==========================================
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.units import mm

def single_txt_to_pdf(text_path, output_path):
    """텍스트 파일 하나를 PDF로 변환"""
    with open(text_path, 'r', encoding='utf-8', errors='replace') as f:
        content = f.read()

    doc = SimpleDocTemplate(
        output_path,
        pagesize=A4,
        leftMargin=20*mm, rightMargin=20*mm,
        topMargin=20*mm, bottomMargin=20*mm
    )

    style = ParagraphStyle(
        'code',
        fontName='HYGothic-Medium',
        fontSize=9,
        leading=14,
        wordWrap='CJK',
    )

    title_style = ParagraphStyle(
        'title',
        fontName='HYGothic-Medium',
        fontSize=13,
        leading=20,
        spaceAfter=6,
    )

    story = []
    filename = os.path.basename(text_path)
    story.append(Paragraph(filename, title_style))
    story.append(Spacer(1, 4*mm))

    for line in content.splitlines():
        safe_line = line.replace('&', '&amp;').replace('<', '&lt;').replace('>', '&gt;')
        story.append(Paragraph(safe_line if safe_line.strip() else '&nbsp;', style))

    doc.build(story)


def process_txt_to_pdf(txt_files, output_mode, custom_name):
    if not txt_files:
        return None, "파일을 업로드해주세요.", gr.update(interactive=False)
    try:
        generated = []
        for f in txt_files:
            name_only = os.path.splitext(os.path.basename(f.name))[0]
            out = f"{name_only}.pdf"
            single_txt_to_pdf(f.name, out)
            generated.append(out)

        if output_mode == "개별 PDF (ZIP으로 묶기)":
            base = custom_name.strip() if custom_name and custom_name.strip() else "txt_to_pdf"
            zip_path = f"{base}.zip"
            with zipfile.ZipFile(zip_path, 'w') as zipf:
                for g in generated:
                    zipf.write(g, os.path.basename(g))
            return generated, f"{len(generated)}개 파일 변환 완료! (ZIP)", gr.update(value=zip_path, interactive=True)

        else:  # 하나의 PDF로 합치기
            merger = PdfWriter()
            for g in generated:
                reader = PdfReader(g)
                for page in reader.pages:
                    merger.add_page(page)
            base = custom_name.strip() if custom_name and custom_name.strip() else "merged"
            out_path = f"{base}.pdf"
            with open(out_path, 'wb') as f:
                merger.write(f)
            return out_path, f"{len(generated)}개 파일 → 1개 PDF 변환 완료!", gr.update(value=out_path, interactive=True)

    except Exception as e:
        return None, f"오류: {str(e)}", gr.update(interactive=False)


# ==========================================
# 2. 오디오 탭 — 구간 분할 / N등분
# ==========================================
def process_audio_trim(audio_file, start_time_str, end_time_str, remove_silence, custom_name):
    if audio_file is None:
        return None, "오디오 파일을 업로드해주세요.", gr.update(interactive=False)
    try:
        audio = AudioSegment.from_file(audio_file)
        start_ms = parse_time_to_ms(start_time_str)
        end_ms = len(audio) if not end_time_str or str(end_time_str).strip() in ["", "0"] else parse_time_to_ms(end_time_str)

        if start_ms >= end_ms and end_ms != len(audio):
            return None, "시작이 종료보다 클 수 없습니다.", gr.update(interactive=False)
        audio = audio[start_ms:end_ms]

        if remove_silence:
            chunks = split_on_silence(audio, min_silence_len=500, silence_thresh=-40)
            if chunks:
                audio = sum(chunks)

        name_only, ext = os.path.splitext(os.path.basename(audio_file))
        ext_lower = ext.lower() if ext.lower() in ['.mp3', '.m4a', '.wav', '.ogg', '.flac'] else '.mp3'
        export_format = "ipod" if ext_lower == ".m4a" else ext_lower.replace('.', '')

        if custom_name and custom_name.strip():
            output_path = f"{custom_name.strip()}{ext_lower}"
        else:
            silence_tag = "_trimmed" if remove_silence else ""
            output_path = f"{name_only}_{start_ms//1000}s_to_{end_ms//1000}s{silence_tag}{ext_lower}"

        audio.export(output_path, format=export_format)

        try:
            orig_meta = mutagen.File(audio_file)
            if orig_meta and orig_meta.tags:
                new_meta = mutagen.File(output_path)
                new_meta.tags = orig_meta.tags
                new_meta.save()
        except Exception:
            pass

        return output_path, "구간 분할 완료!", gr.update(value=output_path, interactive=True)
    except Exception as e:
        return None, f"오류: {str(e)}", gr.update(interactive=False)


def process_audio_nsplit(audio_file, n_splits, custom_name):
    if audio_file is None:
        return None, "오디오 파일을 업로드해주세요.", gr.update(interactive=False)
    try:
        n = int(n_splits)
        if n < 2:
            return None, "2 이상의 숫자를 입력해주세요.", gr.update(interactive=False)

        audio = AudioSegment.from_file(audio_file)
        total_ms = len(audio)
        segment_ms = total_ms // n

        name_only, ext = os.path.splitext(os.path.basename(audio_file))
        ext_lower = ext.lower() if ext.lower() in ['.mp3', '.m4a', '.wav', '.ogg', '.flac'] else '.mp3'
        export_format = "ipod" if ext_lower == ".m4a" else ext_lower.replace('.', '')
        base_name = custom_name.strip() if custom_name and custom_name.strip() else name_only

        output_files = []
        for i in range(n):
            start = i * segment_ms
            end = (i + 1) * segment_ms if i < n - 1 else total_ms
            chunk = audio[start:end]
            chunk_name = f"{base_name}_part{i+1}{ext_lower}"
            chunk.export(chunk_name, format=export_format)
            try:
                orig_meta = mutagen.File(audio_file)
                if orig_meta and orig_meta.tags:
                    new_meta = mutagen.File(chunk_name)
                    new_meta.tags = orig_meta.tags
                    new_meta.save()
            except Exception:
                pass
            output_files.append(chunk_name)

        zip_filename = f"{base_name}_{n}splits.zip"
        with zipfile.ZipFile(zip_filename, 'w') as zipf:
            for f in output_files:
                zipf.write(f, os.path.basename(f))

        return output_files, f"{n}등분 완료!", gr.update(value=zip_filename, interactive=True)
    except Exception as e:
        return None, f"오류: {str(e)}", gr.update(interactive=False)


def handle_audio(mode, audio_file,
                 start_time, end_time, remove_silence,
                 n_splits, custom_name):
    if mode == "구간 분할":
        return process_audio_trim(audio_file, start_time, end_time, remove_silence, custom_name)
    elif mode == "N등분":
        return process_audio_nsplit(audio_file, n_splits, custom_name)
    return None, "기능을 선택해주세요.", gr.update(interactive=False)


# ==========================================
# UI 동적 표시/숨김 헬퍼
# ==========================================
def update_pdf_ui(mode):
    show_split    = gr.update(visible=(mode == "분할"))
    show_merge    = gr.update(visible=(mode == "병합 / 잠금해제"))
    show_security = gr.update(visible=(mode == "보안 / 워터마크"))
    show_txt      = gr.update(visible=(mode == "텍스트 → PDF"))
    return show_split, show_merge, show_security, show_txt
def update_audio_ui(mode):
    show_trim   = gr.update(visible=(mode == "구간 분할"))
    show_nsplit = gr.update(visible=(mode == "N등분"))
    return show_trim, show_nsplit


# ==========================================
# UI 구성
# ==========================================
with gr.Blocks(title="File Swordmaster") as demo:
    gr.HTML("""
    <div style="text-align:center; padding: 24px 0 8px;">
        <h1 style="font-size:2rem; font-weight:700; margin:0;">File Swordmaster</h1>
        <p style="color:#888; margin-top:6px;">PDF · 오디오 파일 처리 도구</p>
    </div>
    """)

    # ── PDF 탭 ──────────────────────────────
    with gr.Tab("PDF"):
        pdf_mode = gr.Radio(
            choices=["분할", "병합 / 잠금해제", "보안 / 워터마크", "텍스트 → PDF"],
            value="분할",
            label="기능 선택",
        )

        with gr.Row():
            # 입력 영역
            with gr.Column(scale=1):

                # 분할 옵션
                with gr.Group(visible=True) as pdf_split_group:
                    gr.Markdown("**분할 옵션**")
                    pdf_single = gr.File(label="PDF 업로드", file_types=[".pdf"])
                    pdf_pages  = gr.Textbox(label="추출 페이지 (예: 1, 3, 5-10)")

                # 병합 옵션
                with gr.Group(visible=False) as pdf_merge_group:
                    gr.Markdown("**병합 옵션**")
                    pdf_multi    = gr.File(label="PDF 여러 개 업로드 (순서대로 병합)", file_types=[".pdf"], file_count="multiple")
                    merge_pw     = gr.Textbox(label="비밀번호 (잠긴 파일)", type="password")

                # 보안 옵션
                with gr.Group(visible=False) as pdf_security_group:
                    gr.Markdown("**보안 / 워터마크 옵션**")
                    security_pdf = gr.File(label="원본 PDF 업로드", file_types=[".pdf"])
                    security_pw  = gr.Textbox(label="새 비밀번호 (공란 시 설정 안 함)", type="password")
                    watermark    = gr.Textbox(label="대각선 워터마크 텍스트", placeholder="예: 대외비, 기밀문서")

                # 텍스트 → PDF 옵션
                with gr.Group(visible=False) as pdf_txt_group:
                    gr.Markdown("**텍스트 → PDF 옵션**")
                    txt_files = gr.File(
                        label="텍스트 파일 업로드 (여러 개 가능)",
                        file_count="multiple",
                        file_types=[".txt", ".md", ".py", ".js", ".ts", ".java", ".c", ".cpp",
                                    ".cs", ".go", ".rs", ".rb", ".php", ".sh", ".yaml", ".yml",
                                    ".json", ".xml", ".html", ".css", ".sql", ".r", ".swift",
                                    ".kt", ".scala", ".toml", ".ini", ".env", ".log"]
                    )
                    txt_output_mode = gr.Radio(
                        choices=["개별 PDF (ZIP으로 묶기)", "하나의 PDF로 합치기"],
                        value="개별 PDF (ZIP으로 묶기)",
                        label="출력 방식"
                    )

                # 공통
                pdf_custom_name = gr.Textbox(label="저장 파일명 (선택, 확장자 생략)", placeholder="자동 생성")
                pdf_btn = gr.Button("실행", variant="primary", size="lg")

            # 출력 영역
            with gr.Column(scale=1):
                pdf_out_file = gr.File(label="결과물", file_count="multiple")
                pdf_out_msg  = gr.Textbox(label="상태", interactive=False)
                pdf_dl_btn   = gr.DownloadButton("다운로드", interactive=False, variant="secondary")

        # 모드 전환 시 UI 업데이트
        pdf_mode.change(
            update_pdf_ui,
            inputs=[pdf_mode],
            outputs=[pdf_split_group, pdf_merge_group, pdf_security_group, pdf_txt_group]
        )

        pdf_btn.click(
            handle_pdf,
            inputs=[pdf_mode, pdf_single, pdf_pages,
                    pdf_multi, merge_pw,
                    security_pdf, security_pw, watermark,
                    txt_files, txt_output_mode,
                    pdf_custom_name],
            outputs=[pdf_out_file, pdf_out_msg, pdf_dl_btn]
        )

    # ── 오디오 탭 ────────────────────────────
    with gr.Tab("오디오"):
        audio_mode = gr.Radio(
            choices=["구간 분할", "N등분"],
            value="구간 분할",
            label="기능 선택",
        )

        with gr.Row():
            with gr.Column(scale=1):
                audio_input = gr.Audio(label="오디오 업로드", type="filepath")

                # 구간 분할 옵션
                with gr.Group(visible=True) as audio_trim_group:
                    gr.Markdown("**구간 분할 옵션**")
                    with gr.Row():
                        audio_start = gr.Textbox(label="시작 시간 (초 또는 분:초)", value="0")
                        audio_end   = gr.Textbox(label="종료 시간 (0이면 끝까지)", value="0")
                    audio_sil = gr.Checkbox(label="무음 구간 자동 제거", value=False)

                # N등분 옵션
                with gr.Group(visible=False) as audio_nsplit_group:
                    gr.Markdown("**N등분 옵션**")
                    n_splits = gr.Number(label="나눌 개수 (N)", value=2, minimum=2, precision=0)

                # 공통
                audio_custom_name = gr.Textbox(label="저장 파일명 (선택)", placeholder="자동 생성")
                audio_btn = gr.Button("실행", variant="primary", size="lg")

            with gr.Column(scale=1):
                audio_out_file = gr.File(label="결과물", file_count="multiple")
                audio_out_msg  = gr.Textbox(label="상태", interactive=False)
                audio_dl_btn   = gr.DownloadButton("다운로드 (ZIP)", interactive=False, variant="secondary")

        audio_mode.change(
            update_audio_ui,
            inputs=[audio_mode],
            outputs=[audio_trim_group, audio_nsplit_group]
        )

        audio_btn.click(
            handle_audio,
            inputs=[audio_mode, audio_input,
                    audio_start, audio_end, audio_sil,
                    n_splits, audio_custom_name],
            outputs=[audio_out_file, audio_out_msg, audio_dl_btn]
        )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://511e6811053e4cf296.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
